# 射电天文有害干扰门限链（RA.769 判据闭式）

消费 `rfauto.core.ra769`（10% 判据 + 辐射计方程 + 干扰几何 margin 链）：从系统噪声温度到『合规/有害』判定的完整闭式链。

- **诚实边界**：逐带 T_sys/Δf/τ 原表数值（RA.769-2 表）在本
  执行环境不可达，本示例全部用**演示参数**（调用方给入口径），
  非 ITU 原表锚定值；
- k_B 取 SI 2019 精确值。

In [ ]:
from rfauto.core.ra769 import (
    detrimental_power_dbw,
    effective_area,
    interference_margin,
    pfd_to_received_power,
    received_power_to_pfd,
)

# 10% 判据：使 T_sys 恶化 10% 的干扰功率门限（τ=2000 s 口径）
crit = detrimental_power_dbw(t_sys_k=50.0, delta_f_hz=1.0e6,
                             tau_s=2000.0)
print(f"ΔT = {crit['delta_t_k']:.1f} K (10% of 50 K)")
print(f"有害干扰门限 P_H = {crit['p_limit_dbw']:.2f} dBW")

In [ ]:
# 有效口径与 PFD↔接收功率（口径定理，闭式互逆）
freq = 1.42e9  # 演示频点
a_e = effective_area(gain_dbi=30.0, freq_hz=freq)
print(f"30 dBi @1.42 GHz → A_e = {a_e:.4f} m²")
p_rx = pfd_to_received_power(pfd_dbw_m2=-180.0, gain_dbi=30.0,
                             freq_hz=freq)
pfd_back = received_power_to_pfd(p_rx_dbw=p_rx, gain_dbi=30.0,
                                 freq_hz=freq)
print(f"PFD=-180 dBW/m² → P_rx={p_rx:.3f} dBW "
      f"→ 回算 PFD={pfd_back:.6f} dBW/m²")
assert abs(pfd_back - (-180.0)) < 1e-9  # 互逆恒等

In [ ]:
# 干扰几何 margin 链（主瓣直射保守分支）
m = interference_margin(eirp_dbw=10.0, distance_m=30.0e3,
                        freq_hz=freq, off_axis_deg=0.0,
                        rx_gain_dbi=30.0,
                        p_limit_dbw=crit["p_limit_dbw"])
print(f"FSPL = {m['fspl_db']:.2f} dB")
print(f"P_rx = {m['p_rx_dbw']:.2f} dBW")
print(f"margin = {m['margin_db']:.2f} dB → {m['verdict']}")